Neccesary libraries for executing the notebook

In [182]:
!pip install pandas
!pip install jsonschema
!pip install nltk
!pip install requests
!pip install openai tabulate

In this first part we are going to preprocces the village-transcript.json to see how the program work.

After that we are going to do the same procces with a bigger file


We read the json to obtain the information

In [ ]:
import json
#read the json 
with open('./json_data/village-transcript.json', 'r', encoding='utf-8') as archivo:
    datos = json.load(archivo)


Convert the json into a dataframe for easier use

In [ ]:
import pandas as pd
events_day = datos["days"][0]["events"]
df = pd.DataFrame(events_day)

Remove data NaN to have a cleaner df and eliminate column duration for it doesn't have use in this case

In [ ]:
def clean_dataframe(df): #cleanind the dataframe to remove unnecessary columns and standardize values
    df_cleaned = df.copy()
    df_cleaned.drop("duration", axis=1, inplace=True)
    df_cleaned.loc[df_cleaned["type"] == "USER_TALK", ["agentName","goal"]] = "no_agent"
    df_cleaned.loc[df_cleaned["type"] == "START_USING_COMPUTER", ["speakerName","content"]] = "agent_starting"
    df_cleaned.loc[df_cleaned["type"] == "AGENT_TALK", ["agentName","goal"]] = "agent_talking"
    df_cleaned.loc[df_cleaned["type"] == "WAIT", ["speakerName","content","goal"]] = "agent_waiting"
    df_cleaned.loc[df_cleaned["type"] == "STOP_USING_COMPUTER", ["speakerName","content","goal"]] = "agent_stopping"
    df_cleaned.loc[df_cleaned["type"] == "PAUSE", ["speakerName","content","goal"]] = "agent_pausing"
    return df_cleaned

In [170]:
df_cleaned = clean_dataframe(df) #execute the cleaning function on the dataframe
df_cleaned.head()

,timestamp,time,type,speakerName,content,agentName,goal
0,2025-04-02T17:47:10.816Z,17:47:10,USER_TALK,zak,Hi everyone! Welcome to the Agent Village.,no_agent,no_agent
1,2025-04-02T17:47:18.073Z,17:47:18,USER_TALK,zak,"Agents, your ultimate goal as part of the vill...",no_agent,no_agent
2,2025-04-02T17:47:42.909Z,17:47:42,USER_TALK,zak,"You each have a computer you can use, but you'...",no_agent,no_agent
3,2025-04-02T17:48:04.344Z,17:48:04,USER_TALK,zak,You can always email me at help@agentvillage.o...,no_agent,no_agent
4,2025-04-02T17:48:45.240Z,17:48:45,USER_TALK,zak,"Lastly, the Agent Village is a publicly access...",no_agent,no_agent


We write the content of the conversation in a txt file to read it later but before writing it we clean it for reduce use of tokens for the API Is

In [171]:
import nltk

# Descarga los paquetes necesarios de NLTK
nltk.download('punkt')
nltk.download('punkt_tab')  # Requerido en versiones recientes de NLTK
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Usuario\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Usuario\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Usuario\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [183]:
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

def clean_and_tokenize(text):
    text = str(text).lower() 
    text = re.sub(r'[^a-z\s]', '', text) #to lower case and remove punctuation
    tokens = word_tokenize(text) 
    stop_words = set(stopwords.words('english'))
    filtered_tokens = [word for word in tokens if word not in stop_words] #tokenise and remove stop words
    return " ".join(filtered_tokens)

  

In [173]:
df_cleaned["tokens"] = df_cleaned["content"].apply(clean_and_tokenize)

We save the content of the conversation in a text file with the format [speakerName]: content

So we can select the messages we want

In [174]:
file_name = "conversation.txt"

with open(file_name, "w", encoding="utf-8") as file:
    for index, row in df_cleaned.iterrows():
        # Escribe con el formato [speakerName]: content
        if row["type"] in ["AGENT_TALK", "USER_TALK"]:
            file.write(f"[{row['speakerName']}]: {row['tokens']}\n")

Now we pass the content of conversation.txt to the AI to give us the final information

In [175]:
with open('conversation.txt', 'r', encoding='utf-8') as archivo:
    conversation = archivo.read()

In [ ]:
from SwarmReporter import SwarmReporter

first_50_lines = "\n".join(conversation.splitlines()[50:100])

reporter = SwarmReporter()
resultados = reporter.generate_report(first_50_lines)
reporter.print_formatted_reports(resultados)

Now that we now it work with village-transcript we are goint to use it for chat_messages too

In [ ]:
df_chat_messages = pd.read_json('./json_data/chat_messages.jsonl', lines=True)
df_chat_messages["tokens"] = df_chat_messages["content"].apply(clean_and_tokenize)

We use send the tokenise text to the api to get the summary

In [ ]:
conversation_long = "\n".join(df_chat_messages["tokens"][0:50].dropna().astype(str)) 
#only use 50 lines to avoid exceeding the token limit of the model
reporter = SwarmReporter()
result = reporter.generate_report(conversation_long)
reporter.print_formatted_reports(result)

Generando informe con modelo: openai/gpt-oss-120b...

INFORME GENERADO POR: openai/gpt-oss-120b

RESUMEN EJECUTIVO:
Los agentes coordinaron la corrección de código, la verificación de repositorios y el despliegue de varios sitios web, mientras gestionaban tareas de outreach, documentación y análisis ético; también monitorizaron el drift del modelo y documentaron cuestiones de procedencia. Se completaron pruebas de funcionalidades de juego y se establecieron procesos de monitoreo continuo. Se acordaron acciones de seguimiento y se esperarán nuevas actualizaciones antes de avanzar en ciertos ítems.

 TEMAS CLAVE:
  • Corrección y verificación de código (commits, PRs, pruebas)
  • Despliegue y verificación de sitios web y dominios
  • Gestión de tareas, seguimiento de progreso y documentación
  • Comunicación externa (email, Instagram, enlaces compartidos)
  • Análisis ético y de procedencia de datos
  • Monitoreo de drift y calidad de IA
  • Pruebas de funcionalidades de juego (quests, f